# FICOS final forensic economic reconciliation

Freshly recompute EXP-04 and EXP-06 from the canonical dataset and executable source formulas. Existing reports and result payloads are not used as primary evidence. Run this notebook on Colab with a T4 runtime if desired.

In [ ]:
import os, sys, subprocess, hashlib
from pathlib import Path
REPO = Path('/content/FICOS-Platform')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/SSOHEB/FICOS-Platform.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pandas','numpy','scikit-learn','pyyaml'], check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
DATA = REPO/'data'/'modeling_dataset.csv'
if not DATA.exists(): raise FileNotFoundError('data/modeling_dataset.csv is unavailable in this checkout')
print('dataset_sha256:', hashlib.sha256(DATA.read_bytes()).hexdigest())
df = pd.read_csv(DATA); df['date'] = pd.to_datetime(df['date']); df = df.sort_values('date').reset_index(drop=True)
VESSELS = ['panamax','supramax','handy','cape']
FOLDS = [
 {'year':2021,'train_end':'2019-12-24','val_start':'2020-01-03','val_end':'2020-12-24','test_start':'2021-01-05','test_end':'2021-12-31'},
 {'year':2022,'train_end':'2020-12-24','val_start':'2021-01-05','val_end':'2021-12-24','test_start':'2022-01-03','test_end':'2022-12-30'},
 {'year':2023,'train_end':'2021-12-24','val_start':'2022-01-03','val_end':'2022-12-23','test_start':'2023-01-03','test_end':'2023-12-29'},
 {'year':2024,'train_end':'2022-12-23','val_start':'2023-01-03','val_end':'2023-12-22','test_start':'2024-01-02','test_end':'2024-12-31'},
 {'year':2025,'train_end':'2023-12-22','val_start':'2024-01-02','val_end':'2024-12-24','test_start':'2025-01-02','test_end':'2025-12-31'}]
FEATURES = [c for c in df.columns if c != 'date' and not c.startswith('target_') and not c.startswith('dir_')]
VOYAGE, IDLE, WAIT_DAYS, TREES, SEED = 20.0, 2500.0, 1.0, 100, 42

In [ ]:
def run_fold(fold, vessel):
    target = f'target_{vessel}_1d'; valid = df[vessel].notna() & df[target].notna()
    masks = [(df.date <= fold['train_end']) & valid, (df.date >= fold['val_start']) & (df.date <= fold['val_end']) & valid, (df.date >= fold['test_start']) & (df.date <= fold['test_end']) & valid]
    def X(m): return np.nan_to_num(df.loc[m, FEATURES].to_numpy(), nan=0, posinf=0, neginf=0)
    def y(m): return df.loc[m,target].to_numpy() - df.loc[m,vessel].to_numpy()
    scaler = StandardScaler().fit(X(masks[0])); selector = SelectKBest(f_regression,k=min(30,len(FEATURES))).fit(scaler.transform(X(masks[0])),y(masks[0]))
    model = RandomForestRegressor(n_estimators=TREES,max_depth=5,random_state=SEED,n_jobs=1).fit(selector.transform(scaler.transform(X(masks[0]))),y(masks[0]))
    val_pred = model.predict(selector.transform(scaler.transform(X(masks[1])))); test_pred = model.predict(selector.transform(scaler.transform(X(masks[2]))))
    val_base, val_true = df.loc[masks[1],vessel].to_numpy(), df.loc[masks[1],target].to_numpy(); test_base, test_true = df.loc[masks[2],vessel].to_numpy(), df.loc[masks[2],target].to_numpy()
    candidates = np.linspace(-300,-25,50); vals = [np.sum(val_base*VOYAGE - np.where(val_pred<t,val_true*VOYAGE+IDLE,val_base*VOYAGE)) for t in candidates]; threshold = float(candidates[int(np.argmax(vals))])
    dates = df.loc[masks[2],'date'].dt.strftime('%Y-%m-%d').to_numpy()
    return threshold, val_pred, test_pred, test_base, test_true, dates

rows, audit = [], []
for fold in FOLDS:
  for vessel in VESSELS:
    threshold, _, pred, base, true, dates = run_fold(fold,vessel)
    audit.append({**fold,'vessel':vessel,'threshold':threshold})
    for i in range(len(true)):
      spot = base[i]*VOYAGE; wait = true[i]*VOYAGE+IDLE*WAIT_DAYS
      for exp, decision in [('EXP-04','WAIT' if pred[i] < -125 else 'SPOT'), ('EXP-06','WAIT' if pred[i] < threshold else 'SPOT_INDEX')]:
        policy = wait if decision == 'WAIT' else spot
        rows.append({'experiment':exp,'date':dates[i],'year':fold['year'],'vessel':vessel,'pred_delta':pred[i],'base_rate':base[i],'true_rate':true[i],'decision':decision,'spot_cost':spot,'policy_cost':policy,'net_savings':spot-policy,'wait_correct':np.sign(true[i]-base[i])==np.sign(pred[i])})
results = pd.DataFrame(rows); audit = pd.DataFrame(audit)
display(audit); display(results.groupby('experiment').agg(observations=('decision','size'),WAIT_N=('decision',lambda x:(x=='WAIT').sum()),baseline=('spot_cost','sum'),policy_cost=('policy_cost','sum'),net_savings=('net_savings','sum')))

In [ ]:
def summary(x):
  w=x[x.decision=='WAIT']; nw=x[x.decision!='WAIT']
  return pd.Series({'observations':len(x),'WAIT_N':len(w),'non_WAIT_N':len(nw),'WAIT_precision':w.wait_correct.mean() if len(w) else np.nan,'gross_WAIT_gains':w.loc[w.net_savings>0,'net_savings'].sum(),'gross_WAIT_losses':w.loc[w.net_savings<0,'net_savings'].sum(),'net_WAIT_value':w.net_savings.sum(),'non_WAIT_contribution':nw.net_savings.sum(),'total_net_savings':x.net_savings.sum()})
for exp in ['EXP-04','EXP-06']:
  print(exp); display(pd.concat([summary(results[results.experiment==exp]),summary(results[(results.experiment==exp)&(results.year==2025)])],axis=1,keys=['all_years','2025']))
# Historical FLEX sensitivity: same predictions and decisions, old formula applied to non-WAIT rows.
x=results[results.experiment=='EXP-06'].copy(); historical=np.where(x.decision=='WAIT',x.true_rate*VOYAGE+IDLE,(x.base_rate+0.5*(x.true_rate-x.base_rate))*VOYAGE+IDLE*0.25)
print('EXP-06 current net:',x.net_savings.sum()); print('EXP-06 with historical FLEX formula:',(x.spot_cost-historical).sum())
OUT=REPO/'outputs'/'forensic_colab'; OUT.mkdir(parents=True,exist_ok=True); results.to_csv(OUT/'fresh_exp04_exp06_observations.csv',index=False); audit.to_csv(OUT/'fresh_walkforward_thresholds.csv',index=False)
print('Fresh evidence written to',OUT)